# Computational Theory: SHA-256

## Introduction

SHA-256 is a hash function from the SHA-2 family, defined in the [Secure Hash Standard (FIPS 180-4)](https://doi.org/10.6028/NIST.FIPS.180-4). It takes a message of any length and returns a 256-bit digest. Even a one-bit change in the message gives a completely different digest, so SHA-256 is used for checking downloaded files, digital signatures and [Bitcoin](https://en.wikipedia.org/wiki/Bitcoin).

In this notebook I build SHA-256 step by step with Python and NumPy instead of just calling [`hashlib`](https://docs.python.org/3/library/hashlib.html). Each problem adds one part:

1. how the data is represented in Python,
2. the bitwise functions,
3. the round constants,
4. padding the message and splitting it into blocks,
5. the compression function,
6. the full `sha256()` function, checked against `hashlib`.

Run the cells from top to bottom, because later problems use functions from earlier ones.

## Problem 1: Representing SHA-256 Data

These examples follow the [SHA-256 standard](https://doi.org/10.6028/NIST.FIPS.180-4).

### 32-bit words

I use [`np.uint32`](https://numpy.org/doc/stable/reference/arrays.scalars.html#unsigned-integer-types) because SHA-256 uses unsigned 32-bit words. Its range is 0 to 2**32 - 1; Python's normal `int` has no fixed size.

In [1]:
import numpy as np

# A 32-bit unsigned word, the basic unit SHA-256 works with.
word = np.uint32(0x12345678)
# The same value in decimal, binary (32 digits) and hex (8 digits).
print("Decimal:", word)
print("Binary:", f"{word:032b}")
print("Hex:", f"{word:08x}")
# A uint32 always takes exactly 4 bytes.
print("Size in bytes:", word.itemsize)

Decimal: 305419896
Binary: 00010010001101000101011001111000
Hex: 12345678
Size in bytes: 4


### Sequences of words

A NumPy array with `dtype=np.uint32` keeps the words in order, with the same type for every element.

In [2]:
# An array keeps several words in order, all as uint32.
words = np.array([1, 2, 3], dtype=np.uint32)
print("Words:", words)
# Every element has the same type.
print("Type:", words.dtype)
# Three words of 4 bytes each.
print("Size in bytes:", words.nbytes)

Words: [1 2 3]
Type: uint32
Size in bytes: 12


### Input messages

I use `bytes` because the input is binary data. Text is [encoded as UTF-8](https://docs.python.org/3/library/stdtypes.html#str.encode); `café` has four characters but five bytes.

In [3]:
text = "café"
# SHA-256 works on bytes, so the text is encoded first.
message = text.encode("utf-8")
print("Message:", message)
# The counts differ because the last character takes two bytes.
print("Characters:", len(text))
print("Bytes:", len(message))

Message: b'caf\xc3\xa9'
Characters: 4
Bytes: 5


### 512-bit message blocks

A block is 64 bytes, split into sixteen big-endian `uint32` words for processing. This is a sample block; padding is covered in Problem 4.

In [4]:
# A sample 64-byte (512-bit) block with the byte values 0 to 63.
block = bytes(range(64))
# Read every 4 bytes as one big-endian word, giving sixteen words.
block_words = np.array(
    [int.from_bytes(block[i:i + 4], "big") for i in range(0, 64, 4)],
    dtype=np.uint32,
)
print("Block bytes:", len(block))
print("Block words:", len(block_words))
# The first word is bytes 00 01 02 03 and the last is 3c 3d 3e 3f.
print("First word:", f"{block_words[0]:08x}")
print("Last word:", f"{block_words[-1]:08x}")

Block bytes: 64
Block words: 16
First word: 00010203
Last word: 3c3d3e3f


### The final 256-bit hash

Eight `uint32` words make 32 bytes. I convert them to 64 lowercase hex characters for readable output. The example values only show the format.

In [5]:
# Eight example words standing in for the final hash state.
hash_words = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=np.uint32)
# Turn each word into 4 big-endian bytes and join them: 8 x 4 = 32 bytes.
digest = b"".join(int(word).to_bytes(4, "big") for word in hash_words)
# Each byte becomes two hex characters, so 32 bytes give 64.
hex_digest = digest.hex()
print("Digest bytes:", len(digest))
print("Hex digest:", hex_digest)
print("Hex characters:", len(hex_digest))

Digest bytes: 32
Hex digest: 0000000100000002000000030000000400000005000000060000000700000008
Hex characters: 64


### Big-endian integers

Big-endian puts the most significant byte first. I use [`int.from_bytes()` and `int.to_bytes()`](https://docs.python.org/3/library/stdtypes.html#int.from_bytes) with `"big"` to follow SHA-256's byte order.

In [6]:
value = 0x12345678
# Big-endian puts the most significant byte (12) first.
big_endian = value.to_bytes(4, "big")
# Little-endian puts it last, so the bytes are reversed.
little_endian = value.to_bytes(4, "little")
print("Big-endian:", big_endian.hex(" "))
print("Little-endian:", little_endian.hex(" "))
# Reading the bytes back in the same order gives the original value.
print("Read back:", hex(int.from_bytes(big_endian, "big")))

Big-endian: 12 34 56 78
Little-endian: 78 56 34 12
Read back: 0x12345678


### Summary

| Data | Python type | Size |
| --- | --- | --- |
| Word | `np.uint32` | 32 bits (4 bytes) |
| Sequence of words | NumPy array with `dtype=np.uint32` | 4 bytes per word |
| Input message | `bytes` | any length |
| Message block | sixteen `np.uint32` words | 512 bits (64 bytes) |
| Final hash | eight `np.uint32` words, shown as hex | 256 bits (32 bytes, 64 hex characters) |
| Byte order | `"big"` in `int.from_bytes()` and `int.to_bytes()` | most significant byte first |

### Research: why SHA-256 uses these formats

I wanted to know why the data is set up like this, so I read a bit more about it.

The word size depends on which hash you use. The table in section 1 of the [standard](https://doi.org/10.6028/NIST.FIPS.180-4) shows that SHA-256 has 32-bit words and 512-bit blocks, while SHA-512 has 64-bit words and 1024-bit blocks. Both have sixteen words in a block, SHA-512 just has bigger ones.

Big-endian is basically just a rule. Section 3.1 of the standard uses big-endian, but the older [MD4 hash](https://www.rfc-editor.org/rfc/rfc1320) does the opposite and puts the lowest byte first. Neither way is better, but you have to stick to one. If I read a word the wrong way round I get a different number, and the hash comes out wrong. This is why I always write `"big"` in my code.

For text, [RFC 3629](https://www.rfc-editor.org/rfc/rfc3629) says UTF-8 uses 1 to 4 bytes per character, and normal ASCII characters stay as one byte. So in `café`, the `c`, `a` and `f` take one byte each and `é` takes two, which is why I got five bytes. SHA-256 only sees bytes, so the same text in another encoding would give a different hash.

## Problem 2: SHA-256 Bitwise Operations

The definitions follow section 4.1.2 of the [Secure Hash Standard](https://doi.org/10.6028/NIST.FIPS.180-4). I use `np.uint32` so every result stays inside 32 bits. The standard writes the operations with maths symbols, and Python has its own symbol for each one:

| Operation | Standard | Python | What it does to each bit |
| --- | --- | --- | --- |
| AND | $\land$ | `&` | 1 only if both bits are 1 |
| XOR | $\oplus$ | `^` | 1 if the two bits are different |
| NOT | $\lnot$ | `~` | flips 0 to 1 and 1 to 0 |
| OR | $\lor$ | <code>&#124;</code> | 1 if at least one bit is 1 |
| Shift right | $\gg$ | `>>` | moves bits right, zeros come in on the left |
| Shift left | $\ll$ | `<<` | moves bits left, zeros come in on the right |

### Ch: choose

$$Ch(x, y, z) = (x \land y) \oplus (\lnot x \land z)$$

In Python this is `(x & y) ^ (~x & z)`.

At each bit position, `x` chooses: if the bit of `x` is 1, the result takes the bit from `y`, and if it is 0, it takes the bit from `z`. A small 4-bit example:

| | bits |
| --- | --- |
| `x` | `1100` |
| `y` | `1010` |
| `z` | `0110` |
| `Ch` | `1010` (first two bits from `y`, last two from `z`) |

With `uint32`, NOT flips exactly 32 bits. The tests cover all eight combinations of all-zero and all-one words, a mixed pattern, and the highest bit.

In [7]:
def Ch(x, y, z):
    """Return a uint32 choosing bits from y or z using x."""
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    # Where x has a 1 take the bit from y, where x has a 0 take it from z.
    return (x & y) ^ (~x & z)

In [8]:
from itertools import product

# Try every mix of all-zero and all-one words: 2 * 2 * 2 = 8 cases.
for x, y, z in product((0, 0xFFFFFFFF), repeat=3):
    result = Ch(x, y, z)
    # If x is all ones the answer is y, otherwise it is z.
    assert result == (y if x else z)
    assert isinstance(result, np.uint32)

# A mixed pattern, where different bits choose differently.
assert Ch(0xF0F0F0F0, 0xCCCCCCCC, 0xAAAAAAAA) == 0xCACACACA
# The highest bit is handled like any other bit.
assert Ch(0x80000000, 0xFFFFFFFF, 0) == 0x80000000
print("Ch tests passed")

Ch tests passed


### Maj: majority

$$Maj(x, y, z) = (x \land y) \oplus (x \land z) \oplus (y \land z)$$

In Python this is `(x & y) ^ (x & z) ^ (y & z)`.

Each output bit is a vote: it is 1 when at least two of the three input bits are 1. With exactly two ones, only one of the three pairs is 1. With three ones, all three pairs are 1, and XOR of three ones is also 1. A small 4-bit example:

| | bits |
| --- | --- |
| `x` | `1100` |
| `y` | `1010` |
| `z` | `0110` |
| `Maj` | `1110` (the last bit has no ones, so it is 0) |

The tests cover all eight combinations of all-zero and all-one words, a mixed pattern, and the highest bit.

In [9]:
def Maj(x, y, z):
    """Return the bitwise majority of three words as uint32."""
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    # A bit is 1 when at least two of the inputs have a 1 there.
    return (x & y) ^ (x & z) ^ (y & z)

In [10]:
# Try every mix of all-zero and all-one words.
for x, y, z in product((0, 0xFFFFFFFF), repeat=3):
    result = Maj(x, y, z)
    # The answer is all ones when at least two of the inputs are all ones.
    expected = 0xFFFFFFFF if (x, y, z).count(0xFFFFFFFF) >= 2 else 0
    assert result == expected
    assert isinstance(result, np.uint32)

# A mixed pattern, where each bit has its own vote.
assert Maj(0xF0F0F0F0, 0xCCCCCCCC, 0xAAAAAAAA) == 0xE8E8E8E8
# Two out of three is enough for the highest bit too.
assert Maj(0x80000000, 0x80000000, 0) == 0x80000000
print("Maj tests passed")

Maj tests passed


### Right rotation

$$ROTR^n(x) = (x \gg n) \lor (x \ll (32 - n))$$

A right rotation moves every bit `n` places to the right, and the bits that fall off the right end come back in on the left. Nothing is lost. For example, rotating `12345678` right by 8 bits moves the last byte `78` to the front and gives `78123456`.

The first part, `x >> n`, is the shifted word. The second part, `x << (32 - n)`, puts the bits that fell off at the left end. I join them with OR and use `np.uint32` so the result stays 32 bits. SHA-256 only uses rotation counts between 1 and 31.

In [11]:
def ROTR(x, n):
    """Rotate a 32-bit word right by 1 to 31 bits."""
    x = np.uint32(x)
    # Shift right by n, and move the n bits that fall off to the left side.
    return (x >> np.uint32(n)) | (x << np.uint32(32 - n))

In [12]:
# Rotating all zeros or all ones changes nothing.
assert ROTR(0, 1) == 0
assert ROTR(0xFFFFFFFF, 1) == 0xFFFFFFFF
# The lowest bit wraps around to the highest bit.
assert ROTR(1, 1) == 0x80000000
# Rotating right by 31 is the same as rotating left by 1.
assert ROTR(1, 31) == 2
# The last byte moves to the front.
assert ROTR(0x12345678, 8) == 0x78123456
assert isinstance(ROTR(1, 1), np.uint32)
print("ROTR tests passed")

ROTR tests passed


### Sigma0

$$\Sigma_0^{256}(x) = ROTR^2(x) \oplus ROTR^{13}(x) \oplus ROTR^{22}(x)$$

I rotate the same word by 2, 13 and 22 bits, then combine the three results with XOR. Every bit of the result depends on three different bits of `x`, which mixes the word up. The rotation counts come from the standard.

In [13]:
def Sigma0(x):
    """Return the SHA-256 upper-case Sigma0 of x."""
    # Three rotations of the same word, joined with XOR.
    return ROTR(x, 2) ^ ROTR(x, 13) ^ ROTR(x, 22)

In [14]:
# Zero stays zero, because rotating and XOR-ing zeros gives zeros.
assert Sigma0(0) == 0
# All ones XOR-ed three times is still all ones.
assert Sigma0(0xFFFFFFFF) == 0xFFFFFFFF
# A single 1 bit ends up in three places.
assert Sigma0(1) == 0x40080400
# A mixed word, checked against plain Python further down.
assert Sigma0(0x12345678) == 0x66146474
assert isinstance(Sigma0(1), np.uint32)
print("Sigma0 tests passed")

Sigma0 tests passed


### Sigma1

$$\Sigma_1^{256}(x) = ROTR^6(x) \oplus ROTR^{11}(x) \oplus ROTR^{25}(x)$$

This works like `Sigma0`, but rotates by 6, 11 and 25 bits. Both upper-case functions are used in every round of the compression function in Problem 5.

In [15]:
def Sigma1(x):
    """Return the SHA-256 upper-case Sigma1 of x."""
    # Same idea as Sigma0, with rotations of 6, 11 and 25.
    return ROTR(x, 6) ^ ROTR(x, 11) ^ ROTR(x, 25)

In [16]:
# Zero stays zero, because rotating and XOR-ing zeros gives zeros.
assert Sigma1(0) == 0
# All ones XOR-ed three times is still all ones.
assert Sigma1(0xFFFFFFFF) == 0xFFFFFFFF
# A single 1 bit ends up in three places.
assert Sigma1(1) == 0x04200080
# A mixed word, checked against plain Python further down.
assert Sigma1(0x12345678) == 0x3561ABDA
assert isinstance(Sigma1(1), np.uint32)
print("Sigma1 tests passed")

Sigma1 tests passed


### sigma0

$$\sigma_0^{256}(x) = ROTR^7(x) \oplus ROTR^{18}(x) \oplus (x \gg 3)$$

I rotate `x` right by 7 and 18 bits and shift it right by 3 bits. Then I XOR the three results. The difference from the upper-case functions is the last part: a shift adds zeros on the left instead of wrapping bits around, so some bits are really lost. The lower-case sigmas are used to build the message schedule in Problem 5.

In [17]:
def sigma0(x):
    """Return sigma0 of a 32-bit word."""
    x = np.uint32(x)
    # The shift adds zeros on the left, it does not wrap around like ROTR.
    return ROTR(x, 7) ^ ROTR(x, 18) ^ (x >> np.uint32(3))

In [18]:
# Zero stays zero, because rotating and XOR-ing zeros gives zeros.
assert sigma0(0) == 0
# The shift brings in 3 zeros, so the top 3 bits are 0.
assert sigma0(0xFFFFFFFF) == 0x1FFFFFFF
# A single 1 bit ends up in two places.
assert sigma0(1) == 0x02004000
# A mixed word, checked against plain Python further down.
assert sigma0(0x12345678) == 0xE7FCE6EE
assert isinstance(sigma0(1), np.uint32)
print("sigma0 tests passed")

sigma0 tests passed


### sigma1

$$\sigma_1^{256}(x) = ROTR^{17}(x) \oplus ROTR^{19}(x) \oplus (x \gg 10)$$

This does the same steps as `sigma0` with different amounts: rotate right by 17 and 19 bits, shift right by 10 bits, then XOR the results.

In [19]:
def sigma1(x):
    """Return sigma1 of a 32-bit word."""
    x = np.uint32(x)
    # Same idea as sigma0, just different amounts.
    return ROTR(x, 17) ^ ROTR(x, 19) ^ (x >> np.uint32(10))

In [20]:
# Zero stays zero, because rotating and XOR-ing zeros gives zeros.
assert sigma1(0) == 0
# The shift brings in 10 zeros, so the top 10 bits are 0.
assert sigma1(0xFFFFFFFF) == 0x003FFFFF
# A single 1 bit ends up in two places.
assert sigma1(1) == 0x0000A000
# A mixed word, checked against plain Python further down.
assert sigma1(0x12345678) == 0xA1F78649
assert isinstance(sigma1(1), np.uint32)
print("sigma1 tests passed")

sigma1 tests passed


### Checking against plain Python

To make sure the expected values in my tests are right, I wrote the same four functions again with plain Python `int` instead of NumPy. Python ints have no fixed size, so I use `& 0xFFFFFFFF` to keep only 32 bits. Then I compare both versions on 1000 random words.

In [21]:
import random


def rotr_int(x, n):
    """Rotate a 32-bit Python int right by n bits."""
    # Cut the result back to 32 bits.
    return ((x >> n) | (x << (32 - n))) & 0xFFFFFFFF


# Same random words every time the notebook runs.
random.seed(0)

for _ in range(1000):
    x = random.getrandbits(32)
    assert Sigma0(x) == rotr_int(x, 2) ^ rotr_int(x, 13) ^ rotr_int(x, 22)
    assert Sigma1(x) == rotr_int(x, 6) ^ rotr_int(x, 11) ^ rotr_int(x, 25)
    assert sigma0(x) == rotr_int(x, 7) ^ rotr_int(x, 18) ^ (x >> 3)
    assert sigma1(x) == rotr_int(x, 17) ^ rotr_int(x, 19) ^ (x >> 10)

print("All four functions match plain Python on 1000 random words")

All four functions match plain Python on 1000 random words


### Research: where these functions come from

I looked up where `Ch` and `Maj` come from, and they are older than SHA-256. The [MD4 hash](https://www.rfc-editor.org/rfc/rfc1320) has the same two ideas: its function `F` works like "if X then Y else Z" and its function `G` is a majority. MD4 joins the parts with OR and SHA-256 uses XOR. For `Ch` this makes no difference, because `x & y` and `~x & z` can never both be 1 on the same bit.

I also found out that the four sigma functions are linear over XOR. That means `Sigma0(a ^ b)` gives the same answer as `Sigma0(a) ^ Sigma0(b)`. They only move bits around and XOR them. `Ch` and `Maj` use AND, so they are not linear. As far as I understand, you need both kinds: the sigmas spread the bits around, and `Ch` and `Maj` stop the whole thing from being simple XOR equations that someone could just solve. [Gilbert and Handschuh](https://doi.org/10.1007/978-3-540-24654-1_13) analysed SHA-256 and showed that a simplified version, with XOR instead of addition and symmetric constants, is weak.

To check this myself I tried it on 1000 random inputs below. The sigmas should pass every time, and `Ch` and `Maj` should almost never pass.

In [22]:
# Same random inputs every time the notebook runs.
random.seed(1)

sigmas = [Sigma0, Sigma1, sigma0, sigma1]
# Count how many times each function passes the linearity check.
linear_counts = {f.__name__: 0 for f in sigmas + [Ch, Maj]}

for _ in range(1000):
    a, b, c, d, e, f = (random.getrandbits(32) for _ in range(6))
    for func in sigmas:
        # Linear means: f(a XOR b) is the same as f(a) XOR f(b).
        linear_counts[func.__name__] += int(func(a ^ b) == func(a) ^ func(b))
    for func in (Ch, Maj):
        # XOR two sets of inputs together and compare with XOR of the outputs.
        linear_counts[func.__name__] += int(func(a ^ d, b ^ e, c ^ f) == func(a, b, c) ^ func(d, e, f))

print(linear_counts)

{'Sigma0': 1000, 'Sigma1': 1000, 'sigma0': 1000, 'sigma1': 1000, 'Ch': 0, 'Maj': 0}


The result is what I expected: all four sigma functions passed all 1000 times, so they really are linear. `Ch` and `Maj` passed 0 times, so they are not. This is the reason SHA-256 needs both kinds of function.

## Problem 3: Generating the SHA-256 Constants

### Prime numbers

A prime number is greater than 1 and has no divisors except 1 and itself. I start at 2 and check each number against the primes already found. `all()` is true when none of them divides the number. I stop after finding `n` primes. This follows the approach suggested in the [course notes](https://github.com/ianmcloughlin/computational-theory/blob/main/notes/prime-numbers.ipynb).

In [23]:
def primes(n):
    """Return the first n prime numbers."""
    numbers = []
    candidate = 2
    while len(numbers) < n:
        # Keep the number if none of the primes so far divide it.
        if all(candidate % p != 0 for p in numbers):
            numbers.append(candidate)
        candidate += 1
    return numbers

In [24]:
# Asking for no primes gives an empty list.
assert primes(0) == []
# The first prime is 2.
assert primes(1) == [2]
# The first ten primes, checked by hand.
assert primes(10) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
# SHA-256 needs 64 primes, and the 64th one is 311.
assert len(primes(64)) == 64
assert primes(64)[-1] == 311
print("primes tests passed")

primes tests passed


### Cube roots

Section 4.2.2 of the standard says each round constant is "the first thirty-two bits of the fractional parts of the cube roots of the first sixty-four prime numbers". In simple steps, for each prime $p$:

1. take the cube root, $\sqrt[3]{p}$,
2. keep only the part after the decimal point,
3. multiply by $2^{32}$ and drop what is left after the point.

For the first prime: $\sqrt[3]{2} = 1.2599\ldots$, the part after the point is $0.2599\ldots$, and $0.2599\ldots \times 2^{32} = 1116352408.8\ldots$, which is `428a2f98` in hex. That is the first constant in the standard.

I find the cube roots of the first 64 primes with `np.cbrt()`. Then I subtract the whole part with `np.floor()` to keep only the part after the decimal point. I print the first five results.

In [25]:
roots = np.cbrt(primes(64))
# Keep only the part after the decimal point.
fractions = roots - np.floor(roots)
print(fractions[:5])

[0.25992105 0.44224957 0.70997595 0.91293118 0.22398009]


### Round constants

I multiply each fraction by `2**32` and convert it to `np.uint32`. This drops the decimal part and gives a 32-bit word. I print each word as eight hex digits, eight words per row, following section 4.2.2 of the [SHA-256 standard](https://doi.org/10.6028/NIST.FIPS.180-4).

In [26]:
# Move the fraction up into 32 bits and drop what is left after the point.
constants = (fractions * 2**32).astype(np.uint32)

# Print eight constants per row, like the table in the standard.
for i in range(0, 64, 8):
    print(" ".join(f"{value:08x}" for value in constants[i:i + 8]))

428a2f98 71374491 b5c0fbcf e9b5dba5 3956c25b 59f111f1 923f82a4 ab1c5ed5
d807aa98 12835b01 243185be 550c7dc3 72be5d74 80deb1fe 9bdc06a7 c19bf174
e49b69c1 efbe4786 0fc19dc6 240ca1cc 2de92c6f 4a7484aa 5cb0a9dc 76f988da
983e5152 a831c66d b00327c8 bf597fc7 c6e00bf3 d5a79147 06ca6351 14292967
27b70a85 2e1b2138 4d2c6dfc 53380d13 650a7354 766a0abb 81c2c92e 92722c85
a2bfe8a1 a81a664b c24b8b70 c76c51a3 d192e819 d6990624 f40e3585 106aa070
19a4c116 1e376c08 2748774c 34b0bcb5 391c0cb3 4ed8aa4a 5b9cca4f 682e6ff3
748f82ee 78a5636f 84c87814 8cc70208 90befffa a4506ceb bef9a3f7 c67178f2


### Checking the constants

I copied the 64 values from section 4.2.2 of the [standard](https://doi.org/10.6028/NIST.FIPS.180-4). I compare them with my results in the same order. All 64 values match.

In [27]:
# The 64 constants copied from section 4.2.2 of the standard.
expected = np.array([
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5,
    0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3,
    0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc,
    0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7,
    0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13,
    0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3,
    0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5,
    0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208,
    0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,
], dtype=np.uint32)

# Same values in the same order means every constant is right.
assert np.array_equal(constants, expected)
print("All 64 match")

All 64 match


### Initial hash values

The standard uses the same trick for the eight starting words of the hash, $H^{(0)}$. Section 5.3.3 says they are the first 32 bits of the fractional parts of the **square** roots of the first eight primes. So I can make them with the same steps, using `np.sqrt()` instead of `np.cbrt()`. I will need these words in Problem 6, where every hash starts from them.

In [28]:
# Square roots of the first 8 primes, keeping only the part after the point.
square_roots = np.sqrt(primes(8))
initial_fractions = square_roots - np.floor(square_roots)

# Move the fraction up into 32 bits, like the round constants.
initial_hash = (initial_fractions * 2**32).astype(np.uint32)

# The eight values from section 5.3.3 of the standard.
expected_initial = np.array([
    0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a,
    0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19,
], dtype=np.uint32)

assert np.array_equal(initial_hash, expected_initial)
print(" ".join(f"{value:08x}" for value in initial_hash))

6a09e667 bb67ae85 3c6ef372 a54ff53a 510e527f 9b05688c 1f83d9ab 5be0cd19


### Research: why cube roots of primes

The constants look random, but anyone can work them out, like I did above. Numbers like this are called [nothing-up-my-sleeve numbers](https://en.wikipedia.org/wiki/Nothing-up-my-sleeve_number). If a designer just picks constants and does not explain them, people can wonder if there is a hidden weakness inside. With cube roots of primes there is not much room to cheat. Other hashes do the same thing: [MD5](https://www.rfc-editor.org/rfc/rfc1321) builds its table from `sin(i)`, and section 5.3.3 of the [standard](https://doi.org/10.6028/NIST.FIPS.180-4) gets the starting hash values of SHA-256 from the square roots of the first eight primes.

It is not a full guarantee though. [Bernstein and others](https://ia.cr/2014/571) wrote about how elliptic curve standards could be manipulated. Their paper is not about SHA-256, but the idea is that a designer with enough "innocent" starting values to pick from could still search for one they like.

I also wondered why floats worked here at all, because they are not exact. [Python floats](https://docs.python.org/3/tutorial/floatingpoint.html) have 53 bits of precision. The biggest cube root here is about 6.8, which takes 3 bits for the whole part, so around 50 bits are left for the fraction and I only need 32. SHA-512 needs the first 64 bits of the cube roots of the first 80 primes (section 4.2.3), and that does not fit in a float. To test this I worked out the 64-bit constants exactly with [`decimal`](https://docs.python.org/3/library/decimal.html) and compared them with the float version.

In [29]:
from decimal import Decimal, getcontext

# Enough digits to get 64 bits right after the decimal point.
getcontext().prec = 60

float_matches = 0
exact_constants = []

for p in primes(80):
    exact_root = Decimal(p) ** (Decimal(1) / 3)
    exact_word = int((exact_root - int(exact_root)) * 2**64)
    exact_constants.append(exact_word)

    float_root = np.cbrt(p)
    float_word = int((float_root - np.floor(float_root)) * 2**64)
    float_matches += exact_word == float_word

# The first SHA-256 constant is the top half of the first SHA-512 constant.
assert exact_constants[0] == 0x428a2f98d728ae22
assert exact_constants[0] >> 32 == constants[0]

print("Exact first word:", f"{exact_constants[0]:016x}")
print("Float matches out of 80:", float_matches)

Exact first word: 428a2f98d728ae22
Float matches out of 80: 0


The first exact word matches the first SHA-512 constant in the standard, and its top half is my first SHA-256 constant. But only 0 out of 80 float versions were fully right. So floats are fine for the 32 bits SHA-256 needs, but not for the 64 bits of SHA-512, which needs exact maths like `decimal`.

One more thing about `primes(n)`. It checks each number against every prime found so far, which would be slow for big inputs. The [Sieve of Eratosthenes](https://en.wikipedia.org/wiki/Sieve_of_Eratosthenes) is faster, but you have to know the upper limit in advance, and here I want the first `n` primes. I only need primes up to 311, so the simple version is fine.

## Problem 4: Padding and Parsing Messages

### Why padding is needed

SHA-256 works on blocks of 512 bits (64 bytes), but a message can be any length. So before hashing, the message is padded until its length is a multiple of 512 bits. Section 5.1.1 of the [standard](https://doi.org/10.6028/NIST.FIPS.180-4) gives the rule. For a message that is $\ell$ bits long:

1. add a single `1` bit,
2. add $k$ zero bits, where $k$ is the smallest number (0 or more) so that $\ell + 1 + k \equiv 448 \pmod{512}$, which means $\ell + 1 + k$ leaves 448 when divided by 512,
3. add $\ell$ itself as a 64-bit big-endian number.

448 + 64 = 512, so after step 3 the length is always a whole number of blocks.

The standard's own example is `abc`. It is 3 bytes, so $\ell = 24$. Then 24 + 1 + 423 = 448, and the 64-bit length makes it 512 bits, which is exactly one block. In hex, the padded block looks like this:

```
61 62 63 80 00 00 00 00 00 00 00 00 00 00 00 00    "abc", then 0x80
00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00    zeros
00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00    zeros
00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 18    zeros, then the length: 0x18 = 24
```

### Padding with bytes

My messages are `bytes`, so the length is always a multiple of 8 bits. This makes the rule simpler:

- the `1` bit and the first seven zero bits fit in one byte, `0x80` (`10000000` in binary),
- the rest of the zeros are whole zero bytes, added until the length is 56 bytes more than a multiple of 64 (56 bytes = 448 bits),
- the length is in **bits**, not bytes, so it is `8 * len(msg)`, written as 8 big-endian bytes with `int.to_bytes()` like in Problem 1.

In [30]:
def pad(msg):
    """Return msg padded to a multiple of 64 bytes, following section 5.1.1."""
    # Save the length in bits before the message changes.
    bit_len = len(msg) * 8
    # 0x80 is a 1 bit followed by seven 0 bits.
    msg = msg + b"\x80"
    # % 64 keeps the number of zero bytes between 0 and 63.
    msg = msg + b"\x00" * ((56 - len(msg)) % 64)
    msg = msg + bit_len.to_bytes(8, "big")
    return msg


### Testing the padding

The first test is the `abc` example from the standard: 3 message bytes, then `0x80`, then 52 zero bytes, then the length 24 in the last 8 bytes. The second test is the empty message, which still needs a full block: `0x80`, 55 zero bytes and a length of 0.

In [31]:
# "abc" is 3 bytes, so it fits in one 64-byte block.
padded = pad(b"abc")
assert len(padded) == 64
# The message, then the 0x80 byte.
assert padded[:4] == b"abc\x80"
# Then zeros up to byte 56.
assert padded[4:56] == bytes(52)
# The last 8 bytes are the length in bits: 3 * 8 = 24.
assert padded[56:] == (24).to_bytes(8, "big")

# An empty message still gets a full block of padding.
assert pad(b"") == b"\x80" + bytes(63)

print("Padding tests passed")


Padding tests passed


### Splitting into blocks

Section 5.2.1 of the [standard](https://doi.org/10.6028/NIST.FIPS.180-4) says the padded message is split into blocks of 512 bits (64 bytes). Each block is then read as sixteen 32-bit words, $M_0$ to $M_{15}$. The first 4 bytes make $M_0$, the next 4 make $M_1$, and so on. The words are big-endian, like everything else in SHA-256, so I read them the same way as the sample block in Problem 1.

I write `block_parse` as a [generator](https://realpython.com/introduction-to-python-generators/). It uses `yield` instead of `return`, so it gives back one block at a time instead of building a list of all of them. The compression function in Problem 5 only needs one block at a time, so a long message never has to be held as a list of blocks in memory. The [course notes on padding](https://github.com/ianmcloughlin/computational-theory/blob/main/notes/padding.ipynb) use the same idea to read a file in 64-byte chunks.

In [32]:
def block_parse(msg):
    """Yield each 512-bit block of the padded msg as sixteen uint32 words."""
    padded = pad(msg)
    # Go through the padded message one 64-byte block at a time.
    for i in range(0, len(padded), 64):
        block = padded[i:i + 64]
        # Read every 4 bytes as one big-endian word, the same way as in Problem 1.
        yield np.array(
            [int.from_bytes(block[j:j + 4], "big") for j in range(0, 64, 4)],
            dtype=np.uint32,
        )


### Testing the blocks

For `abc`, section 5.2.1 and the worked example in the standard show one block where the first word is `61626380` (the bytes `a`, `b`, `c` and `0x80`), words 1 to 14 are zero, and the last word is the length, 24. I also check that `block_parse` really is a generator and that a longer message gives more than one block.

In [33]:
from types import GeneratorType

# Calling a generator function gives a generator, not a list.
assert isinstance(block_parse(b"abc"), GeneratorType)

# "abc" fits in one block.
blocks = list(block_parse(b"abc"))
assert len(blocks) == 1

# Each block is sixteen 32-bit words.
first_block = blocks[0]
assert len(first_block) == 16
assert first_block.dtype == np.uint32

# The first word is "abc" and 0x80, read big-endian.
assert first_block[0] == 0x61626380
# Words 1 to 14 are only padding zeros.
assert not first_block[1:15].any()
# The last word is the length in bits.
assert first_block[15] == 24

# 100 bytes plus padding needs two blocks.
assert len(list(block_parse(bytes(100)))) == 2

print("block_parse tests passed")


block_parse tests passed


### Boundary lengths

After the message, the padding always needs at least 9 bytes: one byte `0x80` and 8 bytes for the length. A block is 64 bytes, so a message fits in one block only if it is at most 64 - 9 = 55 bytes. From 56 bytes there is no room left for the length, so the padding moves into a second block.

So the number of blocks is the message length plus 9, rounded up to a multiple of 64. I test the lengths right at the edges (55, 56, 63, 64, 119, 120), and then every length from 0 to 200 against this rule.

In [34]:
# The last length that fits in one block, and the first that does not.
assert len(list(block_parse(bytes(55)))) == 1
assert len(list(block_parse(bytes(56)))) == 2

# A full block of message still needs a second block for the padding.
assert len(list(block_parse(bytes(63)))) == 2
assert len(list(block_parse(bytes(64)))) == 2

# The same edge one block later.
assert len(list(block_parse(bytes(119)))) == 2
assert len(list(block_parse(bytes(120)))) == 3

# For 56 bytes the second block is only zeros and the length: 56 * 8 = 448 bits.
second = list(block_parse(bytes(56)))[1]
assert not second[:15].any()
assert second[15] == 448

# Every length from 0 to 200: message plus 9 bytes, rounded up to 64.
for n in range(201):
    blocks = list(block_parse(bytes(n)))
    assert len(blocks) == (n + 9 + 63) // 64
    # The length in bits is always in the last two words.
    assert (int(blocks[-1][14]) << 32) | int(blocks[-1][15]) == 8 * n

print("Boundary tests passed")


Boundary tests passed


### Byte order

The brief also asks for the message to be read in the right byte order. The sample block from Problem 1 was read as big-endian words by hand, so if I pass the same 64 bytes to `block_parse`, the first block should have exactly the same sixteen words. I also check that the bytes `00 00 00 01` become the word 1, not `0x01000000`, which is what little-endian would give.

In [35]:
# The first block should match the words read by hand in Problem 1.
assert np.array_equal(next(block_parse(block)), block_words)

# Big-endian: the last of the four bytes is the lowest one.
first = next(block_parse(b"\x00\x00\x00\x01"))
assert first[0] == 1
assert first[0] != 0x01000000

print("Byte order tests passed")


Byte order tests passed


### Research: why the padding looks like this

I wanted to know why the padding needs both the `1` bit and the length, and not just zeros. If I only added zero bytes, `abc` and `abc` followed by one zero byte would pad to exactly the same block, so they would always get the same hash. That would be a collision for free. The `1` bit marks where the message really ends, so trailing zeros in the message can't be confused with padding. I check this with a wrong zero-only padding below.

Adding the length at the end is called Merkle–Damgård strengthening. SHA-256 is built the [Merkle–Damgård way](https://en.wikipedia.org/wiki/Merkle%E2%80%93Damg%C3%A5rd_construction): the message is cut into blocks and each block is mixed into the hash state one after another. [Damgård](https://doi.org/10.1007/0-387-34805-0_21) and [Merkle](https://doi.org/10.1007/0-387-34805-0_40) showed in 1989 that, if the padding includes the length, a collision of the whole hash would need a collision in the compression function of a single block. So the hash is as strong as the part I build in Problem 5. The 64-bit length field is also why section 5.1.1 only allows messages shorter than $2^{64}$ bits.

The padding does not fix everything. Because the final hash is just the state after the last block, someone who knows `sha256(secret + message)` and the length can carry on hashing more blocks without knowing the secret. This is called a [length extension attack](https://en.wikipedia.org/wiki/Length_extension_attack). It is why message authentication uses [HMAC (RFC 2104)](https://www.rfc-editor.org/rfc/rfc2104) instead of just hashing a secret with the message.

In [36]:
def zero_pad(msg):
    """Pad msg to a multiple of 64 bytes with zero bytes only (a wrong padding)."""
    return msg + bytes(-len(msg) % 64)


# With zeros only, two different messages give the same padded bytes.
assert zero_pad(b"abc") == zero_pad(b"abc\x00")

# The real padding keeps them different, because of the 0x80 byte and the length.
assert pad(b"abc") != pad(b"abc\x00")

print("Zero-only padding collides, the real padding does not")


Zero-only padding collides, the real padding does not


## Problem 5: The SHA-256 Compression Function

### Overview

The compression function is the main part of SHA-256. It takes two things:

- the current hash state: eight 32-bit words, $H_0$ to $H_7$,
- one message block: sixteen 32-bit words from `block_parse`.

It mixes them together and gives back a new hash state of eight words. For the first block the state is $H^{(0)}$ from Problem 3. Each next block starts from the state the last block gave back.

Section 6.2.2 of the [standard](https://doi.org/10.6028/NIST.FIPS.180-4) splits it into four steps, and I build it in the same order:

| Step | What happens | Uses |
| --- | --- | --- |
| 1. Message schedule | stretch the 16 block words into 64 words, $W_0$ to $W_{63}$ | `sigma0`, `sigma1` |
| 2. Working variables | copy the hash state into eight variables, $a$ to $h$ | $H^{(i-1)}$ |
| 3. 64 rounds | mix one word $W_t$ and one constant $K_t$ into the variables each round | `Ch`, `Maj`, `Sigma0`, `Sigma1`, the round constants |
| 4. Add to the state | add $a$ to $h$ onto the old hash state | addition |

### Addition modulo $2^{32}$

Almost every step above adds words together. In SHA-256 every addition is "modulo $2^{32}$": if the answer is too big for 32 bits, only the lowest 32 bits are kept. It works like a car odometer that goes from 999999 back to 000000. The standard writes this addition as $+$.

NumPy's `uint32` does this on its own, because it only has room for 32 bits. With plain Python `int` there is no limit, so I would have to cut the result back with `& 0xFFFFFFFF`, like in the plain Python check in Problem 2. One thing I noticed: adding two single `np.uint32` values still gives the right answer, but NumPy prints a `RuntimeWarning` about overflow. Adding inside an array does not warn.

In [37]:
import warnings

# The biggest 32-bit word plus 1 wraps round to 0 inside a uint32 array.
biggest = np.array([0xFFFFFFFF], dtype=np.uint32)
print("Array:", biggest + np.uint32(1))

# With plain Python ints I cut the result back to 32 bits myself.
print("Int:", (0xFFFFFFFF + 1) & 0xFFFFFFFF)

# Two single uint32 values also wrap, but NumPy warns about the overflow.
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    result = np.uint32(0xFFFFFFFF) + np.uint32(1)
print("Scalar:", result, "| warning:", caught[0].message)

Array: [0]
Int: 0
Scalar: 0 | warning: overflow encountered in scalar add


### The message schedule

A block only has 16 words, but there are 64 rounds, and each round needs its own word. So step 1 stretches the 16 words into 64. The first 16 are just the block itself. Every word after that is made from four earlier words:

$$
W_t =
\begin{cases}
M_t & 0 \le t \le 15 \\
\sigma_1^{256}(W_{t-2}) + W_{t-7} + \sigma_0^{256}(W_{t-15}) + W_{t-16} & 16 \le t \le 63
\end{cases}
$$

So $W_{16}$ uses $W_{14}$, $W_9$, $W_1$ and $W_0$, then $W_{17}$ uses $W_{15}$, $W_{10}$, $W_2$ and $W_1$, and so on. This is where the lower-case `sigma0` and `sigma1` from Problem 2 are used. They make sure every bit of the block ends up affecting many later words, so changing one bit of the message changes most of the 64 words.

### Building the schedule

I start with an array of 64 zero words and copy the 16 block words to the front. Then I fill in words 16 to 63 one by one with the formula above. Each new word only needs earlier words, so going in order always works.

For the addition I turn the four words into plain Python `int` values, add them, and keep only the lowest 32 bits with `& 0xFFFFFFFF`. This is exactly addition modulo $2^{32}$. It also avoids the NumPy overflow warning from the addition example, and it is the same trick I used in the plain Python check in Problem 2.

In [38]:
def message_schedule(block):
    """Return the 64-word message schedule W for one block of sixteen words."""
    W = np.zeros(64, dtype=np.uint32)
    # The first 16 words are the block itself.
    W[:16] = block
    for t in range(16, 64):
        # Add as plain Python ints, so NumPy does not warn about overflow.
        total = (int(sigma1(W[t - 2])) + int(W[t - 7])
                 + int(sigma0(W[t - 15])) + int(W[t - 16]))
        # Keep only the lowest 32 bits: addition modulo 2**32.
        W[t] = total & 0xFFFFFFFF
    return W


### Testing the message schedule

I test it on the `abc` block from Problem 4. The first 16 words must be the block itself. For $W_{16}$ I can work it out by hand: $W_{14}$, $W_9$ and $W_1$ are all zero for `abc`, and both sigmas of zero are zero, so $W_{16} = W_0$ = `61626380`. For a few later words I worked out the values with plain Python ints outside the notebook. The full `abc` hash at the end checks all 64 words again.

The last test uses a block full of `0xFFFFFFFF`, so almost every addition overflows. It turns warnings into errors, so it fails if the code still gives the overflow warning from the addition example above.

In [39]:
# The block for "abc" from Problem 4.
abc_block = next(block_parse(b"abc"))
W = message_schedule(abc_block)

# 64 words, all 32-bit.
assert len(W) == 64
assert W.dtype == np.uint32

# The first 16 words are just the block.
assert np.array_equal(W[:16], abc_block)

# W14, W9 and W1 are zero for "abc", so W16 is the same as W0.
assert W[16] == 0x61626380

# Later words, worked out with plain Python ints outside the notebook.
assert W[17] == 0x000F0000
assert W[18] == 0x7DA86405
assert W[19] == 0x600003C6
assert W[63] == 0x12B1EDEB

# A block of the biggest words makes the additions overflow.
with warnings.catch_warnings():
    # Turn any warning into an error, so the test fails on overflow warnings.
    warnings.simplefilter("error")
    message_schedule(np.full(16, 0xFFFFFFFF, dtype=np.uint32))

print("Message schedule tests passed")

Message schedule tests passed


## Problem 6: Complete SHA-256